# Лабораторная работа №8
## Тема: Сверточные нейронные сети (CNN) для распознавания изображений

**Цель работы:** изучить принципы построения и работы сверточных нейронных сетей, научиться реализовывать CNN для классификации изображений на примере набора данных CIFAR-10.

**Задачи работы:**
1. Изучить архитектуру сверточной нейронной сети.
2. Освоить основные слои CNN: сверточный, субдискретизации (Pooling), полносвязный.
3. Построить и обучить модель CNN на наборе данных CIFAR-10.
4. Выполнить классификацию произвольного изображения.
5. Выполнить индивидуальное задание и задания для самостоятельной работы.

## 1. Теоретические сведения

Классические полносвязные нейронные сети плохо приспособлены для обработки изображений, так как при векторизации теряются пространственные взаимосвязи между пикселями, а число параметров сети становится очень большим.

**Сверточная нейронная сеть (Convolutional Neural Network, CNN)** — это специальная архитектура нейронной сети, ориентированная на обработку данных, имеющих сеточную структуру (изображения). 

Основное отличие CNN от полносвязных сетей заключается в использовании операции свёртки, позволяющей автоматически выделять локальные признаки: границы, углы, текстуры, формы объектов.

### 1.1. Основные слои CNN

- **Сверточный слой (Conv2D)** – осуществляет свёртку входного изображения с набором фильтров (ядер). В результате формируются карты признаков (feature maps).
- **Функция активации ReLU** – вносит нелинейность в модель: $f(x)=\max(0,x)$.
- **Слой подвыборки (MaxPooling2D)** – уменьшает пространственный размер карт признаков, снижает вычислительную сложность и повышает устойчивость сети к небольшим смещениям.
- **Dropout** – слой регуляризации, случайным образом «отключающий» часть нейронов в процессе обучения для предотвращения переобучения.
- **Полносвязный слой (Dense)** – объединяет извлечённые признаки для принятия итогового решения о принадлежности изображения к тому или иному классу.
- **Softmax** – на выходном слое преобразует значения в вероятности классов, в сумме дающие 1.

## 2. Ход работы

### 2.1. Подключение необходимых библиотек

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing import image

%matplotlib inline

### 2.2. Загрузка и подготовка данных CIFAR-10

Набор данных **CIFAR-10** содержит 60000 цветных изображений размером 32х32 пикселя, разбитых на 10 классов: airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck.

In [ ]:
# Загрузка датасета CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print('Размер обучающей выборки:', x_train.shape)
print('Размер тестовой выборки:', x_test.shape)
print('Размер меток обучающей выборки:', y_train.shape)
print('Размер меток тестовой выборки:', y_test.shape)

### 2.3. Визуализация примеров изображений

In [ ]:
# Список классов CIFAR-10
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Вывод первых 10 изображений
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.suptitle('Примеры изображений из набора CIFAR-10')
plt.show()

### 2.4. Нормировка данных и представление меток в формате one-hot

In [ ]:
# Нормировка пикселей в диапазон [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Преобразование меток в формат one-hot encoding
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

print('Форма x_train:', x_train.shape)
print('Форма y_train_cat:', y_train_cat.shape)

### 2.5. Построение модели сверточной нейронной сети

In [ ]:
# Построение CNN
model = Sequential()

# Блок 1
model.add(Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(32,32,3)))
model.add(Conv2D(32, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# Блок 2
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(Conv2D(64, (3,3), activation='relu', padding='same'))
model.add(MaxPooling2D(pool_size=(2,2)))
model.add(Dropout(0.25))

# Полносвязные слои
model.add(Flatten())
model.add(Dense(512, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

# Компиляция модели
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

### 2.6. Обучение модели

In [ ]:
# Обучение сети
history = model.fit(x_train, y_train_cat,
                    batch_size=64,
                    epochs=15,
                    validation_data=(x_test, y_test_cat),
                    verbose=1)

### 2.7. Оценка модели и графики обучения

In [ ]:
# Оценка точности на тестовой выборке
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print('Точность на тестовом наборе:', round(test_acc*100, 2), '%')
print('Потери на тестовом наборе:', round(test_loss, 4))

# Построение графиков
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Точность на обучении')
plt.plot(history.history['val_accuracy'], label='Точность на валидации')
plt.title('График точности')
plt.xlabel('Эпоха')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Потери на обучении')
plt.plot(history.history['val_loss'], label='Потери на валидации')
plt.title('График функции потерь')
plt.xlabel('Эпоха')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### 2.8. Классификация изображения лошади (horse.jpg)

In [ ]:
# Загрузка и предобработка изображения
img_path = 'horse.jpg'
img = image.load_img(img_path, target_size=(32,32))
img_array = image.img_to_array(img)
img_array = img_array / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Визуализация
plt.imshow(image.load_img(img_path))
plt.title('Исходное изображение для распознавания')
plt.axis('off')
plt.show()

# Предсказание
pred = model.predict(img_array)
pred_class = np.argmax(pred)
print('Предсказанный класс:', class_names[pred_class])
print('Вероятности по классам:')
for i in range(10):
    print(f'{class_names[i]:12s}: {pred[0][i]*100:.2f}%')

## 3. Индивидуальное задание

Построить и обучить сверточную нейронную сеть для классификации изображений набора данных **CIFAR-10** по следующему варианту:

1. **Изменить архитектуру сети.** Добавить третий сверточный блок, состоящий из двух слоёв `Conv2D` с 128 фильтрами и слоя `MaxPooling2D`.
2. **Изменить параметры обучения.** Обучить модель в течение **20 эпох** с размером батча `batch_size=32`.
3. **Сравнить результаты.** Сравнить точность полученной модели с моделью, построенной в пункте 2.5–2.7. Сделать вывод о влиянии углубления сети на качество классификации.

**Результат представить в виде:** изменённого фрагмента кода построения и обучения модели, графиков обучения, значения точности на тестовой выборке и краткого вывода.

## 4. Задания для самостоятельной работы

Выполнить **любые два** задания на выбор:

**Задание 1.** Исследовать влияние функции оптимизации. Сравнить работу модели при использовании оптимизаторов `adam`, `rmsprop`, `sgd`. Для каждого случая построить графики точности и привести итоговое значение `accuracy` на тестовой выборке.

**Задание 2.** Применить аугментацию данных с помощью `ImageDataGenerator` (повороты, сдвиги по горизонтали, отражения). Обучить модель с аугментацией и оценить её эффективность по сравнению с базовой моделью.

**Задание 3.** Исследовать влияние слоя `Dropout`. Построить три варианта сети: без `Dropout`, с `Dropout` 0.25, с `Dropout` 0.5. Сравнить точность на обучающей и тестовой выборках, сделать вывод о переобучении.

**Задание 4.** Провести классификацию собственного изображения. Подготовить любое цветное изображение размером не менее 32х32, привести его к формату CIFAR-10 (32х32) и определить, к какому из 10 классов модель отнесёт его. Проанализировать полученные вероятности.

## 5. Контрольные вопросы

1. В чём основное отличие сверточной нейронной сети от полносвязной?
2. Для чего используется операция свёртки в CNN?
3. Какова роль слоя MaxPooling?
4. Почему при построении CNN применяется слой Dropout?
5. Что такое формат one-hot encoding и для чего он используется при решении задачи многоклассовой классификации?

## 6. Отчёт по работе

В отчёте по лабораторной работе необходимо отразить:

1. Тему, цель и задачи работы.
2. Краткое описание архитектуры построенной CNN.
3. Результаты обучения: графики accuracy/loss, значение точности на тестовой выборке.
4. Результат классификации изображения `horse.jpg`.
5. Выполнение индивидуального задания с пояснениями и выводами.
6. Выполнение заданий для самостоятельной работы.
7. Ответы на контрольные вопросы.
8. Общий вывод по работе.